# Fielded TREC retrieval benchmark

Research-only evaluation of public trial records. Do not upload patient data, FHIR bundles, AWS credentials, or any private clinical data. In Kaggle settings, enable GPU and Internet.

Create these Kaggle Secrets from short-lived S3 pre-signed **GET** URLs: `TRIAL_MATCHER_SOURCE_URL`, `TREC_PART_1_URL` through `TREC_PART_5_URL`, `TREC_TOPICS_URL`, and `TREC_QRELS_URL`.

In [ ]:
import subprocess
subprocess.run(['nvidia-smi'], check=True)
subprocess.run(['python', '--version'], check=True)

In [ ]:
import os
import subprocess
import urllib.request
from urllib.error import HTTPError
from pathlib import Path

from kaggle_secrets import UserSecretsClient

os.environ['TOKENIZERS_PARALLELISM'] = 'false'
workdir = Path('/kaggle/working')
project_root = workdir / 'trial-matcher-backend'
raw_dir = project_root / 'datasets/evaluation/trec/raw'
raw_dir.mkdir(parents=True, exist_ok=True)
secrets = UserSecretsClient()

required = {
    'TRIAL_MATCHER_SOURCE_URL': workdir / 'trial-matcher-source.tgz',
    **{f'TREC_PART_{part}_URL': raw_dir / f'ClinicalTrials.2021-04-27.part{part}.zip' for part in range(1, 6)},
    'TREC_TOPICS_URL': raw_dir / 'topics-2022.xml',
    'TREC_QRELS_URL': raw_dir / 'qrels-2022.txt',
}

for secret_name, destination in required.items():
    url = secrets.get_secret(secret_name)
    if not url:
        raise RuntimeError(f'Missing Kaggle Secret: {secret_name}')
    print(f'Downloading {destination.name}')
    try:
        urllib.request.urlretrieve(url, destination)
    except HTTPError as error:
        detail = error.read().decode('utf-8', errors='replace')[:500]
        raise RuntimeError(
            f'S3 rejected {secret_name} with HTTP {error.code}: {detail}'
        ) from error

subprocess.run(['tar', '-xzf', str(workdir / 'trial-matcher-source.tgz'), '-C', str(project_root)], check=True)
print('Public source and corpus are ready.')

In [ ]:
# Keep this benchmark isolated from Kaggle's preinstalled packages. The
# managed image cannot create a venv, so packages are placed in a dedicated
# folder that is first on PYTHONPATH for every benchmark subprocess.
import shutil
benchmark_packages = workdir / '.trec-benchmark-packages'
shutil.rmtree(benchmark_packages, ignore_errors=True)
benchmark_packages.mkdir()
subprocess.run(
    [
        'python', '-m', 'pip', 'install', '--quiet', '--target', str(benchmark_packages),
        'torch==2.5.1',
        '--index-url', 'https://download.pytorch.org/whl/cu121',
    ],
    check=True,
)
subprocess.run(
    [
        'python', '-m', 'pip', 'install', '--quiet', '--target', str(benchmark_packages),
        'sentence-transformers==3.4.1', 'transformers==4.46.3',
        'SQLAlchemy>=2.0,<3', 'pgvector>=0.3,<1', 'pydantic>=2,<3',
    ],
    check=True,
)
environment = {**os.environ, 'PYTHONPATH': f'{benchmark_packages}:{project_root}'}
verification = (
    "import torch; from sentence_transformers import SentenceTransformer; "
    "assert torch.cuda.is_available(); assert 'sm_60' in torch.cuda.get_arch_list(); "
    "print(torch.__version__, torch.cuda.get_device_name(0))"
)
subprocess.run(['python', '-c', verification], env=environment, check=True)
model_download = (
    "from huggingface_hub import snapshot_download; snapshot_download("
    "repo_id='NeuML/pubmedbert-base-embeddings', "
    "revision='b79526d6ef3645e0df4530322e266f24c829f5ef')"
)
subprocess.run(['python', '-c', model_download], env=environment, check=True)
print('Isolated P100-compatible benchmark environment is ready.')

In [ ]:
environment = {**os.environ, 'PYTHONPATH': str(project_root)}
subprocess.run(
    [
        'python', 'scripts/build_trec_semantic_index.py',
        '--output-dir', 'datasets/evaluation/trec/semantic-fielded',
        '--batch-size', '128',
    ],
    cwd=project_root,
    env=environment,
    check=True,
)

In [ ]:
report = workdir / 'trec-fielded-benchmark.json'
subprocess.run(
    [
        'python', 'scripts/evaluate_trec_hybrid.py',
        '--semantic-dir', 'datasets/evaluation/trec/semantic-fielded',
        '--output', str(report),
    ],
    cwd=project_root,
    env=environment,
    check=True,
)

import json
results = json.loads(report.read_text())
print(json.dumps({
    'document_profile': results['document_profile'],
    'field_weights': results['field_weights'],
    'semantic': results['semantic']['metrics'],
    'hybrid': results['hybrid']['metrics'],
}, indent=2))
print(f'Saved report: {report}')